# Chapter 2 — Pre-Model Workflow and Data Preprocessing
**Buku:** *scikit-learn Cookbook, Third Edition* (John Sukup, Packt, 2025)
**Cakupan notebook:** reproduksi kode + ringkasan + penjelasan teori Chapter 2



## Ringkasan Chapter
Kualitas data adalah faktor terpenting keberhasilan model ML (*garbage in, garbage out*), dan preprocessing adalah pekerjaan berkelanjutan, bukan sekali jalan. Chapter 2 membahas alat preprocessing scikit-learn: **penanganan missing data** (`SimpleImputer`, `KNNImputer`, `IterativeImputer`), **scaling** (`StandardScaler`, `MinMaxScaler`, `Normalizer`), **encoding variabel kategorikal** (`OneHotEncoder`, `LabelEncoder`, `ColumnTransformer`), **`Pipeline`**, dan **feature engineering** (`PolynomialFeatures`, `KBinsDiscretizer`, `RFE`, `SelectFromModel`). Ditutup dengan latihan membangun pipeline lengkap pada dataset California Housing.

## 0. Dampak Data Mentah terhadap Performa Model
Model belajar pola dari data. Bila data cacat, kemampuan generalisasi ke data baru menurun. Masalah data yang umum:

| Masalah | Dampak | Alat scikit-learn |
|---|---|---|
| **Missing data** | Banyak algoritma gagal atau membuat asumsi keliru | `SimpleImputer`, `KNNImputer`, `IterativeImputer` |
| **Outlier** | Mendistorsi statistik dan melatih model secara bias | `RobustScaler`, analisis z-score |
| **Variabel kategorikal** | Kebanyakan algoritma butuh input numerik | `OneHotEncoder`, `LabelEncoder` |
| **Skala fitur berbeda** | Mengganggu konvergensi dan model berbasis jarak (KNN, SVM) | `StandardScaler`, `MinMaxScaler` |
| **Data leakage** | Informasi data uji "bocor" ke pelatihan sehingga performa tampak terlalu bagus | split dulu, lalu `Pipeline` |

Alat bantu umum: `Pipeline` (transformasi konsisten), `ColumnTransformer` (preprocessing berbeda untuk kolom berbeda), dan `sklearn.feature_selection` (`SelectFromModel`, `RFE`).

In [1]:
import warnings
warnings.filterwarnings("ignore")   # menyembunyikan warning agar notebook rapi

import numpy as np
import pandas as pd
from IPython.display import display
import sklearn
print("scikit-learn:", sklearn.__version__)

scikit-learn: 1.8.0


## 1. Handling Missing Data
Data hilang bisa berasal dari kesalahan manusia, kegagalan teknis, atau data rusak. Kebanyakan algoritma scikit-learn **tidak bisa berjalan** bila ada `NaN`. Opsinya: membuang baris (aman bila data besar) atau **imputasi**, yaitu mengisi nilai hilang dengan estimasi dari data yang ada.

### Getting ready: membuat dataset mainan
20 sampel, 10 fitur acak (0–100), dengan sekitar 20% nilai dihapus secara acak. `np.random.seed(2024)` memastikan hasil dapat direproduksi.

In [2]:
# [Dari buku] Dataset mainan dengan missing values
np.random.seed(2024)          # reproducibility
n_samples = 20
n_features = 10

data = {f"Feature{i+1}": np.random.uniform(0, 100, n_samples) for i in range(n_features)}
df = pd.DataFrame(data)

# Hapus ~20% nilai secara acak
for column in df.columns:
    mask = np.random.random(n_samples) < 0.2
    df.loc[mask, column] = np.nan

display(df)

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.801452,NaN,42.009814,34.680397,49.962259,NaN,NaN,89.954588,41.152421,NaN
1,69.910875,9.554215,6.436369,31.287816,37.966499,NaN,82.005649,NaN,75.797616,91.382492
2,NaN,96.090974,59.643269,84.710402,NaN,84.109027,NaN,70.288128,1.778343,4.117690
3,NaN,25.176729,83.732372,88.023110,16.886931,97.205554,84.696823,NaN,NaN,80.077973
4,20.501895,NaN,89.248639,67.655865,58.635861,78.225721,60.911562,NaN,65.114243,99.119187
5,10.606287,76.825393,20.052744,5.367515,NaN,19.703051,34.423301,93.399494,72.206680,12.640276
6,72.724014,79.792340,50.239523,55.921377,6.191019,NaN,22.966899,21.049022,57.358544,14.302591
7,NaN,NaN,89.538184,69.451294,NaN,47.885551,NaN,33.620401,99.685711,6.683138
8,47.384570,NaN,25.592093,82.419730,73.414540,61.663700,29.172571,65.946718,61.005155,34.052747
9,44.829582,38.165095,NaN,31.142866,28.865545,NaN,41.004459,41.460336,50.236236,NaN


In [3]:
# [Tambahan] Berapa banyak missing value per kolom?
print(df.isna().sum())
print("Total NaN:", int(df.isna().sum().sum()), "dari", df.size, "sel",
      f"({df.isna().sum().sum() / df.size:.1%})")

Feature1     3
Feature2     5
Feature3     4
Feature4     4
Feature5     5
Feature6     5
Feature7     7
Feature8     6
Feature9     2
Feature10    3
dtype: int64
Total NaN: 44 dari 200 sel (22.0%)


### 1a. `SimpleImputer`
Mengganti nilai hilang dengan statistik sederhana per kolom: `mean`, `median`, `most_frequent`, atau `constant`. Cepat dan ringan, cocok bila proporsi data hilang kecil.

In [4]:
# [Dari buku] SimpleImputer (strategy="mean")
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="mean")
imputed_data = imputer.fit_transform(df)
imputed_df = pd.DataFrame(imputed_data, columns=df.columns)
imputed_df.head(6)

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.801452,53.864661,42.009814,34.680397,49.962259,57.822964,51.145011,89.954588,41.152421,48.400044
1,69.910875,9.554215,6.436369,31.287816,37.966499,57.822964,82.005649,50.715003,75.797616,91.382492
2,52.558605,96.090974,59.643269,84.710402,46.055883,84.109027,51.145011,70.288128,1.778343,4.117690
3,52.558605,25.176729,83.732372,88.023110,16.886931,97.205554,84.696823,50.715003,60.616723,80.077973
4,20.501895,53.864661,89.248639,67.655865,58.635861,78.225721,60.911562,50.715003,65.114243,99.119187
5,10.606287,76.825393,20.052744,5.367515,46.055883,19.703051,34.423301,93.399494,72.206680,12.640276


In [5]:
# [Tambahan] Verifikasi: sel Feature1 yang tadinya NaN kini berisi mean kolom
print("Mean Feature1 (tanpa NaN):", round(df["Feature1"].mean(), 6))
print(imputed_df["Feature1"].head(4).round(6).tolist())
print("Sisa NaN:", int(imputed_df.isna().sum().sum()))

Mean Feature1 (tanpa NaN): 52.558605
[58.801452, 69.910875, 52.558605, 52.558605]
Sisa NaN: 0


**Interpretasi:** setiap `NaN` pada suatu kolom diganti dengan rata-rata kolom tersebut (di buku, 52.558605 untuk `Feature1`). Kelemahannya: nilai pengganti sama untuk semua baris sehingga mengabaikan hubungan antarfitur dan mengecilkan variansi.

### 1b. `KNNImputer`
Untuk setiap nilai hilang, mencari `n_neighbors` sampel terdekat di ruang fitur (jarak dihitung dari fitur yang tersedia) lalu memakai nilai tetangga tersebut. Untuk data numerik, nilai yang diisi adalah **rata-rata nilai fitur itu pada tetangga terdekat** (bukan "suara mayoritas" seperti pada klasifikasi KNN). Mampu menangkap hubungan yang lebih halus antar-sampel.

In [6]:
# [Dari buku] KNNImputer
from sklearn.impute import KNNImputer

knn_imputer = KNNImputer(n_neighbors=2)
knn_imputed_data = knn_imputer.fit_transform(df)
knn_imputed_df = pd.DataFrame(knn_imputed_data, columns=df.columns)
knn_imputed_df.head(6)

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.801452,48.954043,42.009814,34.680397,49.962259,93.910386,52.549271,89.954588,41.152421,59.833678
1,69.910875,9.554215,6.436369,31.287816,37.966499,86.793468,82.005649,45.416191,75.797616,91.382492
2,67.752344,96.090974,59.643269,84.710402,73.338309,84.109027,59.012876,70.288128,1.778343,4.117690
3,47.880864,25.176729,83.732372,88.023110,16.886931,97.205554,84.696823,64.510281,39.726833,80.077973
4,20.501895,31.670912,89.248639,67.655865,58.635861,78.225721,60.911562,25.903612,65.114243,99.119187
5,10.606287,76.825393,20.052744,5.367515,59.716773,19.703051,34.423301,93.399494,72.206680,12.640276


### 1c. `IterativeImputer`
Memodelkan tiap fitur yang memiliki `NaN` sebagai fungsi regresi dari fitur-fitur lain secara bergiliran (*round-robin*): fitur target = `y`, fitur lain = `X`. Model regresi lalu memprediksi nilai yang hilang, dan proses diulang. Hasilnya memanfaatkan hubungan multivariat, tetapi lebih berat secara komputasi. Fitur ini masih **experimental**, sehingga harus diaktifkan dulu.

In [7]:
# [Dari buku] IterativeImputer (experimental, harus di-enable dulu)
from sklearn.experimental import enable_iterative_imputer   # noqa: F401
from sklearn.impute import IterativeImputer

iterative_imputer = IterativeImputer()
iterative_imputed_data = iterative_imputer.fit_transform(df)
iterative_imputed_df = pd.DataFrame(iterative_imputed_data, columns=df.columns)
iterative_imputed_df.head(6)

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.801452,54.365008,42.009814,34.680397,49.962259,58.680582,51.178102,89.954588,41.152421,48.292275
1,69.910875,9.554215,6.436369,31.287816,37.966499,60.070634,82.005649,50.710323,75.797616,91.382492
2,52.539676,96.090974,59.643269,84.710402,46.121290,84.109027,51.155371,70.288128,1.778343,4.117690
3,52.630591,25.176729,83.732372,88.023110,16.886931,97.205554,84.696823,50.680265,50.650877,80.077973
4,20.501895,53.198954,89.248639,67.655865,58.635861,78.225721,60.911562,50.691674,65.114243,99.119187
5,10.606287,76.825393,20.052744,5.367515,46.156310,19.703051,34.423301,93.399494,72.206680,12.640276


In [8]:
# [Tambahan] Bandingkan ketiga imputer pada satu sel yang sama (Feature1, baris 1 dan 2)
cmp = pd.DataFrame({
    "original": df["Feature1"],
    "simple_mean": imputed_df["Feature1"],
    "knn": knn_imputed_df["Feature1"],
    "iterative": iterative_imputed_df["Feature1"],
}).loc[df["Feature1"].isna()]
display(cmp.round(3))

,original,simple_mean,knn,iterative
2,NaN,52.559,67.752,52.540
3,NaN,52.559,47.881,52.631
7,NaN,52.559,47.630,52.558


**Interpretasi:** pada baris yang sama, `SimpleImputer` memberi nilai yang identik (mean), sedangkan `KNNImputer` dan `IterativeImputer` memberi nilai yang berbeda-beda per baris karena memakai informasi fitur lain. Karena dataset ini acak murni (tidak ada hubungan antarfitur), perbedaan kualitas tidak bisa dinilai di sini; pada data nyata dengan korelasi antarfitur, metode multivariat biasanya lebih akurat.

### There's more…: memilih strategi imputasi
- **Tipe data:** numerik → mean/median; kategorikal → modus (`most_frequent`).
- **Distribusi data:** distribusi miring/ada outlier → median lebih tahan banting.
- **Pola missing:** acak total (MCAR) atau tidak acak sangat memengaruhi pilihan.
- **Proporsi missing (panduan buku):** sedikit → `SimpleImputer`; sedang (±5–10%) → `KNNImputer` (semua numerik) atau `IterativeImputer` (tipe campuran); sangat banyak (>10%) pada satu fitur → pertimbangkan membuang fitur tersebut.

> **Catatan koreksi:** di buku, kalimat pertama panduan proporsi tertulis "*>5%*" untuk `SimpleImputer`. Dari konteks (metode ringan dan bias kecil), maksudnya kemungkinan besar **<5%**, jadi saya menuliskannya "sedikit".

Tidak semua model butuh imputasi: **decision tree dan Random Forest** dapat menangani missing values (pada versi scikit-learn terbaru).

## 2. Scaling Techniques
Fitur dengan skala berbeda (misal usia 0–100 vs gaji 0–100.000) mengganggu algoritma yang sensitif skala seperti KNN, SVM, dan model berbasis gradient descent. Scaling menjamin tidak ada fitur yang mendominasi.

**Dua istilah yang sering tertukar:**
- **Standardization (z-score):** mengubah data menjadi mean 0 dan standar deviasi 1.
- **Normalization:** mengubah rentang data menjadi 0–1 (pada buku, termasuk `Normalizer` yang bekerja per baris).

Recipe ini memakai `iterative_imputed_df` dari recipe sebelumnya.

### 2a. `StandardScaler`
$$z = \frac{x - \mu}{\sigma}$$
dengan $\mu$ = mean fitur dan $\sigma$ = standar deviasi. Cocok bila data kira-kira berdistribusi normal.

In [9]:
# [Dari buku] StandardScaler
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled_data = scaler.fit_transform(iterative_imputed_df)
scaled_df = pd.DataFrame(scaled_data, columns=iterative_imputed_df.columns)
scaled_df.head(6)

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,0.271931,0.019495,-0.373057,-0.894258,0.188761,0.031844,0.001905,1.683800,-0.780898,-0.003574
1,0.756048,-1.874939,-1.592196,-1.043864,-0.391744,0.085949,1.426871,-0.000534,0.652484,1.429056
2,-0.000939,1.783513,0.231259,1.311954,0.002887,1.021588,0.000854,0.839730,-2.409929,-1.472256
3,0.003022,-1.214477,1.056818,1.458037,-1.411839,1.531340,1.551267,-0.001824,-0.387917,1.053212
4,-1.397054,-0.029802,1.245866,0.559887,0.608499,0.792594,0.451822,-0.001334,0.210478,1.686279
5,-1.828276,0.969036,-1.125549,-2.186891,0.004582,-1.485267,-0.772566,1.831653,0.503915,-1.188904


### 2b. `MinMaxScaler`
$$x' = \frac{x - x_{min}}{x_{max} - x_{min}}$$
Mengubah setiap fitur ke rentang [0, 1] dan mempertahankan bentuk distribusi. Sensitif terhadap outlier karena min/max ikut berubah.

In [10]:
# [Dari buku] MinMaxScaler
from sklearn.preprocessing import MinMaxScaler

minmax_scaler = MinMaxScaler()
minmax_scaled_data = minmax_scaler.fit_transform(iterative_imputed_df)
minmax_scaled_df = pd.DataFrame(minmax_scaled_data, columns=iterative_imputed_df.columns)
minmax_scaled_df.head(6)

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,0.603506,0.517824,0.380540,0.354639,0.569020,0.555896,0.435006,0.962767,0.402156,0.464988
1,0.721357,0.000000,0.000369,0.313594,0.413077,0.571920,0.833074,0.538604,0.756013,0.918562
2,0.537080,1.000000,0.568987,0.959922,0.519088,0.849027,0.434713,0.750206,0.000000,0.000000
3,0.538045,0.180530,0.826426,1.000000,0.139045,1.000000,0.867825,0.538279,0.499171,0.799569
4,0.197218,0.504349,0.885377,0.753589,0.681776,0.781206,0.560692,0.538402,0.646896,1.000000
5,0.092244,0.777371,0.145886,0.000000,0.519543,0.106575,0.218656,1.000000,0.719336,0.089710


### 2c. `Normalizer`
Berbeda dari dua scaler sebelumnya (yang bekerja **per kolom**), `Normalizer` bekerja **per baris (sampel)**, menskalakan setiap sampel agar memiliki panjang (*norm*) = 1.
$$\|x\|_1 = \sum_i |x_i| \qquad \|x\|_2 = \sqrt{\sum_i x_i^2}$$
Berguna untuk data *sparse* atau bila besar-kecilnya sampel tidak penting (misalnya vektor teks). Default `norm="l2"`.

In [11]:
# [Dari buku] Normalizer (default: L2)
from sklearn.preprocessing import Normalizer

normalizer = Normalizer()
normalized_data = normalizer.fit_transform(iterative_imputed_df)
normalized_df = pd.DataFrame(normalized_data, columns=iterative_imputed_df.columns)
normalized_df.head(6)

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,0.339168,0.313579,0.242313,0.200037,0.288183,0.338471,0.295196,0.518860,0.237368,0.278551
1,0.376706,0.051482,0.034682,0.168591,0.204578,0.323683,0.441878,0.273247,0.408426,0.492404
2,0.264336,0.483450,0.300075,0.426192,0.232044,0.423166,0.257371,0.353631,0.008947,0.020717
3,0.243762,0.116607,0.387811,0.407684,0.078213,0.450213,0.392278,0.234729,0.234592,0.370886
4,0.095909,0.248868,0.417511,0.316499,0.274302,0.365945,0.284948,0.237139,0.304609,0.463686
5,0.068115,0.493382,0.128781,0.034471,0.296421,0.126535,0.221071,0.599823,0.463720,0.081177


In [12]:
# [Tambahan] Bukti properti masing-masing scaler
print("StandardScaler  -> mean per kolom ≈ 0 :", scaled_df.mean().abs().max().round(10),
      "| std per kolom ≈ 1 :", scaled_df.std(ddof=0).round(6).unique())
print("MinMaxScaler    -> min =", minmax_scaled_df.min().min(), "| max =", minmax_scaled_df.max().max())
print("Normalizer      -> panjang tiap BARIS (L2) :", np.linalg.norm(normalized_df, axis=1).round(6)[:5], "...")

StandardScaler  -> mean per kolom ≈ 0 : 0.0 | std per kolom ≈ 1 : [1.]
MinMaxScaler    -> min = 0.0 | max = 1.0000000000000002
Normalizer      -> panjang tiap BARIS (L2) : [1. 1. 1. 1. 1.] ...


**Interpretasi:** `StandardScaler` menghasilkan mean 0 dan std 1 per kolom (nilai bisa negatif), `MinMaxScaler` memampatkan setiap kolom ke 0–1, dan `Normalizer` membuat panjang vektor tiap baris = 1.

### There's more…: pengaruh scaling per algoritma
| Algoritma | Efek scaling |
|---|---|
| KNN | Sangat sensitif karena jarak dipengaruhi skala fitur |
| SVM | Sensitif terhadap magnitudo fitur; bekerja baik dengan data terstandardisasi |
| Regresi linear | Konvergensi lebih cepat dengan fitur terstandardisasi |
| Decision tree / Random Forest | **Tidak butuh scaling** karena split berdasarkan urutan nilai |

## 3. Encoding Categorical Variables
Komputer dan sebagian besar algoritma ML membutuhkan input numerik. Variabel kategorikal ada dua jenis:
- **Nominal:** tanpa urutan (warna, merek, departemen).
- **Ordinal:** ada urutan (rating 1–5, Junior < Senior < Manager).

### Getting ready: dataset kategorikal mainan

In [13]:
# [Dari buku] Dataset kategorikal mainan (20 record)
# Catatan: buku hanya mengimpor numpy di sel ini tetapi memakai pd; pandas sudah diimpor di atas.
np.random.seed(2024)
categories = ["A", "B", "C", "D"]
categorical_data = pd.DataFrame({
    "Department": np.random.choice(categories, size=20),
    "Position": np.random.choice(["Junior", "Senior", "Manager"], size=20),
    "Location": np.random.choice(["NY", "SF", "LA", "CHI"], size=20),
})
display(categorical_data)

,Department,Position,Location
0,A,Manager,LA
1,C,Manager,LA
2,A,Junior,NY
3,A,Manager,LA
4,D,Manager,NY
5,A,Senior,LA
6,C,Manager,SF
7,D,Manager,LA
8,B,Junior,LA
9,D,Manager,CHI


### 3a. `OneHotEncoder`
Membuat satu kolom biner per kategori (1 = kategori itu, 0 = bukan). Cocok untuk **nominal** karena setiap kategori diperlakukan independen. Kekurangan: banyak kategori → banyak kolom, matriks besar dan *sparse*.

In [14]:
# [Dari buku] OneHotEncoder
from sklearn.preprocessing import OneHotEncoder

onehot_encoder = OneHotEncoder(sparse_output=False)
onehot_encoded_data = onehot_encoder.fit_transform(categorical_data)
onehot_encoded_df = pd.DataFrame(onehot_encoded_data, columns=onehot_encoder.get_feature_names_out())
display(onehot_encoded_df.head(6))
print("Bentuk:", categorical_data.shape, "->", onehot_encoded_df.shape)

,Department_A,Department_B,Department_C,Department_D,Position_Junior,Position_Manager,Position_Senior,Location_CHI,Location_LA,Location_NY,Location_SF
0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
1,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
2,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
4,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
5,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0


Bentuk: (20, 3) -> (20, 11)


**Interpretasi:** 3 kolom asli menjadi 4 + 3 + 4 = **11 kolom biner** (misalnya `Location_LA`). Setiap baris memiliki tepat satu angka `1` per kelompok fitur asal.

### 3b. `LabelEncoder`
Memberi bilangan bulat unik per kategori (contoh: Red=0, Green=1, Blue=2). Ringkas, tetapi **memperkenalkan urutan semu** yang bisa menyesatkan algoritma yang menganggap angka berarti jarak atau peringkat.

In [15]:
# [Dari buku] LabelEncoder dipakai per kolom
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
label_encoded_df = pd.DataFrame()
for column in categorical_data.columns:
    label_encoded_df[f"{column}_encoded"] = label_encoder.fit_transform(categorical_data[column])
display(label_encoded_df.head(6))

,Department_encoded,Position_encoded,Location_encoded
0,0,1,1
1,2,1,1
2,0,0,2
3,0,1,1
4,3,1,2
5,0,2,1


In [16]:
# [Tambahan] Alternatif yang tepat untuk FITUR ordinal: OrdinalEncoder dengan urutan eksplisit
from sklearn.preprocessing import OrdinalEncoder

ord_enc = OrdinalEncoder(categories=[["Junior", "Senior", "Manager"]])
pos_ord = ord_enc.fit_transform(categorical_data[["Position"]])
print(pd.DataFrame({"Position": categorical_data["Position"], "ordinal": pos_ord.ravel()}).drop_duplicates().sort_values("ordinal"))

  Position  ordinal
2   Junior      0.0
5   Senior      1.0
0  Manager      2.0


> **Catatan penting:** secara resmi, `LabelEncoder` dirancang untuk mengubah **variabel target (`y`)**, bukan fitur. Untuk **fitur**, gunakan `OneHotEncoder` (nominal) atau `OrdinalEncoder` (ordinal, dengan urutan kategori yang kita tentukan sendiri, seperti sel tambahan di atas). `LabelEncoder` pada buku dipakai agar perbedaan antarteknik terlihat. Di `LabelEncoder`, urutan angka mengikuti urutan alfabet (Junior=0, Manager=1, Senior=2), yang tidak sesuai urutan jabatan sebenarnya.

### 3c. `ColumnTransformer`
Pada data campuran (numerik + kategorikal), `ColumnTransformer` menerapkan preprocessing **berbeda untuk kolom berbeda** dalam satu objek: misalnya `StandardScaler` untuk kolom numerik dan `OneHotEncoder` untuk kolom kategorikal.

In [17]:
# [Dari buku] Dataset campuran
np.random.seed(2024)
mixed_data = pd.DataFrame({
    "Age": np.random.randint(25, 65, size=20),
    "Salary": np.round(np.random.normal(60000, 15000, size=20), 2),
    "Experience": np.random.randint(1, 20, size=20),
    "Department": np.random.choice(["IT", "HR", "Sales", "Finance"], size=20),
    "Position": np.random.choice(["Junior", "Senior", "Manager"], size=20),
})
display(mixed_data)

,Age,Salary,Experience,Department,Position
0,33,59420.36,12,Finance,Manager
1,57,82895.92,17,Sales,Junior
2,25,38165.76,16,Finance,Manager
3,52,38242.36,7,IT,Junior
4,61,55088.65,8,Sales,Manager
5,26,78688.88,9,Sales,Senior
6,60,49585.62,14,Finance,Junior
7,35,47992.58,17,HR,Manager
8,27,54833.02,9,Sales,Senior
9,57,93535.56,14,Sales,Senior


In [18]:
# [Dari buku] ColumnTransformer: scaler untuk numerik, one-hot untuk kategorikal
from sklearn.compose import ColumnTransformer

column_transformer = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), ["Age", "Salary", "Experience"]),
        ("cat", OneHotEncoder(), ["Department", "Position"]),
    ],
    remainder="passthrough",
)
transformed_data = column_transformer.fit_transform(mixed_data)

numeric_cols = ["Age_scaled", "Salary_scaled", "Experience_scaled"]
categorical_cols = column_transformer.named_transformers_["cat"].get_feature_names_out(["Department", "Position"])

transformed_df = pd.DataFrame(transformed_data, columns=numeric_cols + list(categorical_cols))
display(transformed_df.head(6))
print("Bentuk:", mixed_data.shape, "->", transformed_df.shape)

,Age_scaled,Salary_scaled,Experience_scaled,Department_Finance,Department_HR,Department_IT,Department_Sales,Position_Junior,Position_Manager,Position_Senior
0,-1.045349,-0.305043,0.327303,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.727681,1.105091,1.262454,0.0,0.0,0.0,1.0,1.0,0.0,0.0
2,-1.636358,-1.581768,1.075424,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,0.358300,-1.577167,-0.607848,0.0,0.0,1.0,0.0,1.0,0.0,0.0
4,1.023186,-0.565241,-0.420818,0.0,0.0,0.0,1.0,0.0,1.0,0.0
5,-1.562482,0.852382,-0.233788,0.0,0.0,0.0,1.0,0.0,0.0,1.0


Bentuk: (20, 5) -> (20, 10)


**Interpretasi:** 3 kolom numerik diskalakan (z-score), 2 kolom kategorikal diubah menjadi 4 + 3 = 7 kolom biner, total **10 kolom**. Argumen `remainder="passthrough"` meloloskan kolom yang tidak disebut (di sini tidak ada). Urutan kolom hasil mengikuti urutan `transformers`.

## 4. Introduction to Pipelines in scikit-learn
`Pipeline` merangkai langkah preprocessing dan model menjadi satu objek, dieksekusi berurutan. Setiap langkah berupa pasangan `(nama, objek)`; **langkah terakhir boleh berupa estimator** (model), sedangkan langkah sebelumnya harus transformer.

Keuntungan: **eksekusi berurutan** yang konsisten, **kode ringkas**, **konsistensi** antara fase latih dan prediksi (mengurangi risiko data leakage), **tuning hyperparameter** sekaligus untuk preprocessing dan model, serta **modularitas**.

Sintaks umum:
```python
pipeline = Pipeline([("nama_langkah", transformer), ..., ("nama_langkah", estimator)])
```

In [19]:
# [Dari buku] Pisahkan fitur dan target, lalu split SEBELUM transformasi apa pun
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

# Untuk demonstrasi, buku memakai kolom terakhir transformed_df sebagai target
X = transformed_df.iloc[:, :-1]   # semua kolom kecuali yang terakhir
y = transformed_df.iloc[:, -1]    # kolom terakhir (Position_Senior) sebagai target contoh

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2024)
print("Train:", X_train.shape, "| Test:", X_test.shape)

Train: (16, 9) | Test: (4, 9)


In [20]:
# [Dari buku] Pipeline: imputer -> scaler
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),   # tangani missing values
    ("scaler", StandardScaler()),                  # skalakan fitur
])

X_train_transformed = pipeline.fit_transform(X_train)   # fit + transform pada TRAIN
X_test_transformed = pipeline.transform(X_test)         # hanya transform pada TEST

# DataFrame agar nama kolom tetap terjaga
X_train_transformed = pd.DataFrame(X_train_transformed, columns=X_train.columns, index=X_train.index)
X_test_transformed = pd.DataFrame(X_test_transformed, columns=X_test.columns, index=X_test.index)
display(X_train_transformed.head())

,Age_scaled,Salary_scaled,Experience_scaled,Department_Finance,Department_HR,Department_IT,Department_Sales,Position_Junior,Position_Manager
13,0.834298,-0.867451,-1.559779,-0.57735,-0.480384,2.081666,-0.774597,-0.57735,-1.0
12,-0.689202,1.282818,-0.150946,-0.57735,2.081666,-0.480384,-0.774597,-0.57735,1.0
16,0.544107,-1.173357,-1.761041,-0.57735,-0.480384,2.081666,-0.774597,-0.57735,1.0
5,-1.342131,0.821525,-0.150946,-0.57735,-0.480384,-0.480384,1.290994,-0.57735,-1.0
17,1.414679,0.611115,0.654101,-0.57735,-0.480384,-0.480384,1.290994,-0.57735,1.0


> **Catatan:** target di sini hanya contoh agar alur terlihat, tanpa makna bisnis (kolom terakhir hasil one-hot). Langkah `imputer` juga tidak benar-benar mengubah apa pun karena `transformed_df` tidak punya `NaN`; ia tetap disertakan untuk menunjukkan cara merangkai dua langkah.

### Mengapa split dulu, baru transformasi?
- **Menghindari data leakage:** statistik (mean, std, dll.) hanya dihitung dari data latih, sehingga tidak ada informasi data uji yang "bocor" ke pelatihan.
- **Evaluasi realistis:** data uji mensimulasikan data baru yang belum pernah dilihat.
- **Konsistensi:** parameter transformasi dipelajari dari data latih lalu dipakai ulang persis pada data uji.
- **Efisiensi:** setelah `fit` pipeline, `predict()` otomatis menerapkan semua transformasi.
- **Mencegah overfitting** akibat karakteristik data uji ikut terpelajari.

In [21]:
# [Tambahan] Bukti: mean yang dipakai scaler berasal dari TRAIN saja
print("Mean yang dipelajari scaler (train):", pipeline.named_steps["scaler"].mean_.round(3))
print("Mean fitur asli di TEST            :", X_test.mean().values.round(3))

Mean yang dipelajari scaler (train): [-0.196 -0.017 -0.094  0.25   0.188  0.188  0.375  0.25   0.5  ]
Mean fitur asli di TEST            : [0.783 0.066 0.374 0.5   0.25  0.25  0.    1.    0.   ]


### Visualisasi pipeline
`set_config(display="diagram")` menampilkan pipeline sebagai diagram interaktif.

In [22]:
# [Dari buku] Visualisasi pipeline
from sklearn import set_config
set_config(display="diagram")
pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...), ('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'mean'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a fea

## 5. Feature Engineering
*Feature engineering* adalah payung untuk dua aktivitas utama:
1. **Feature extraction (pembuatan fitur):** mengubah data yang ada menjadi variabel baru yang menangkap pola/relasi (contoh: *total spending* = harga × jumlah).
2. **Feature selection (pemilihan fitur):** mempertahankan fitur informatif dan membuang yang tidak berguna.

Hasilnya: model lebih sederhana, generalisasi lebih baik, overfitting berkurang. Pendekatan seleksi fitur secara umum ada tiga: **filter** (berdasarkan sifat statistik fitur), **wrapper** (mengevaluasi subset fitur dengan melatih model), dan **hybrid** (gabungan keduanya).

Recipe ini memakai `X_train_transformed` dari recipe pipeline.

### 5a. `PolynomialFeatures`
Membangkitkan fitur polinomial dan interaksi. Untuk `degree=2` dari fitur $a, b$ dihasilkan: $1, a, b, a^2, ab, b^2$.

In [23]:
# [Dari buku] PolynomialFeatures(degree=2)
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2)
poly_features = poly.fit_transform(X_train_transformed)
poly_features_df = pd.DataFrame(poly_features, columns=poly.get_feature_names_out(X_train_transformed.columns))
display(poly_features_df.head())
print("Fitur asli:", X_train_transformed.shape[1], "-> setelah polynomial:", poly_features_df.shape[1])

,1,Age_scaled,Salary_scaled,Experience_scaled,Department_Finance,Department_HR,Department_IT,Department_Sales,Position_Junior,Position_Manager,...,Department_IT^2,Department_IT Department_Sales,Department_IT Position_Junior,Department_IT Position_Manager,Department_Sales^2,Department_Sales Position_Junior,Department_Sales Position_Manager,Position_Junior^2,Position_Junior Position_Manager,Position_Manager^2
0,1.0,0.834298,-0.867451,-1.559779,-0.57735,-0.480384,2.081666,-0.774597,-0.57735,-1.0,...,4.333333,-1.612452,-1.20185,-2.081666,0.600000,0.447214,0.774597,0.333333,0.57735,1.0
1,1.0,-0.689202,1.282818,-0.150946,-0.57735,2.081666,-0.480384,-0.774597,-0.57735,1.0,...,0.230769,0.372104,0.27735,-0.480384,0.600000,0.447214,-0.774597,0.333333,-0.57735,1.0
2,1.0,0.544107,-1.173357,-1.761041,-0.57735,-0.480384,2.081666,-0.774597,-0.57735,1.0,...,4.333333,-1.612452,-1.20185,2.081666,0.600000,0.447214,-0.774597,0.333333,-0.57735,1.0
3,1.0,-1.342131,0.821525,-0.150946,-0.57735,-0.480384,-0.480384,1.290994,-0.57735,-1.0,...,0.230769,-0.620174,0.27735,0.480384,1.666667,-0.745356,-1.290994,0.333333,0.57735,1.0
4,1.0,1.414679,0.611115,0.654101,-0.57735,-0.480384,-0.480384,1.290994,-0.57735,1.0,...,0.230769,-0.620174,0.27735,-0.480384,1.666667,-0.745356,1.290994,0.333333,-0.57735,1.0


Fitur asli: 9 -> setelah polynomial: 55


**Interpretasi:** 9 fitur berkembang menjadi **55 fitur** (1 bias + 9 asli + 45 suku kuadrat dan interaksi). Jumlah fitur tumbuh cepat seiring `degree`, jadi risiko overfitting naik. Biasanya diikuti seleksi fitur atau regularisasi.

### 5b. `KBinsDiscretizer`
Mengubah variabel kontinu menjadi kategorikal dengan memasukkannya ke "keranjang" (*bin*). Jumlah keranjang diatur `n_bins`, dan `strategy` menentukan cara membaginya:
- `uniform`: lebar bin sama
- `quantile`: jumlah data per bin sama
- `kmeans`: bin berdasarkan pusat klaster

In [24]:
# [Dari buku] KBinsDiscretizer
from sklearn.preprocessing import KBinsDiscretizer

kbins = KBinsDiscretizer(n_bins=3, encode="ordinal", strategy="uniform")
binned_data = kbins.fit_transform(X_train_transformed)
binned_df = pd.DataFrame(binned_data, columns=X_train_transformed.columns)
display(binned_df.head())

,Age_scaled,Salary_scaled,Experience_scaled,Department_Finance,Department_HR,Department_IT,Department_Sales,Position_Junior,Position_Manager
0,2.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0
1,0.0,2.0,1.0,0.0,2.0,0.0,0.0,0.0,2.0
2,2.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,2.0
3,0.0,2.0,1.0,0.0,0.0,0.0,2.0,0.0,0.0
4,2.0,1.0,2.0,0.0,0.0,0.0,2.0,0.0,2.0


**Interpretasi:** setiap nilai kini menjadi 0, 1, atau 2 (indeks bin) per fitur. Informasi detail hilang, tetapi hubungan non-linear sederhana dapat lebih mudah ditangkap model.

### 5c. Seleksi fitur: `RFE` (Recursive Feature Elimination)
Awalnya model dilatih dengan semua fitur memakai estimator yang memberi bobot kepentingan fitur. Fitur **paling tidak penting dibuang (*pruning*)**, model dilatih ulang, dan diulang. `ranking_` berisi peringkat tiap fitur (1 = paling penting). Ada juga `RFECV` yang menambahkan cross-validation.

In [33]:
# [Dari buku] RFE dengan LinearRegression
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

rfe = RFE(estimator=LinearRegression(), n_features_to_select=1)
rfe.fit(X_train_transformed, y_train)

print("ranking_:", rfe.ranking_)          # di buku: [5 4 8 6 9 3 7 2 1]; hasil bisa berbeda, lihat catatan di bawah
display(pd.Series(rfe.ranking_, index=X_train_transformed.columns, name="ranking").sort_values())

ranking_: [7 5 4 3 6 9 8 2 1]


Position_Manager      1
Position_Junior       2
Department_Finance    3
Experience_scaled     4
Salary_scaled         5
Department_HR         6
Age_scaled            7
Department_Sales      8
Department_IT         9
Name: ranking, dtype: int64

**Interpretasi:** semakin kecil angkanya, semakin lama fitur bertahan saat *pruning* (peringkat 1 = fitur terakhir yang tersisa = paling penting menurut model). Hasil peringkat bergantung pada data dan estimator yang dipakai.


### 5d. Seleksi fitur: `SelectFromModel`
Memilih fitur berdasarkan **bobot kepentingan** dari sebuah model (`coef_` pada model linear atau `feature_importances_` pada model berbasis pohon). Dengan `threshold="mean"`, fitur dipertahankan bila kepentingannya (nilai mutlak) di atas rata-rata.

In [34]:
# [Dari buku] SelectFromModel dengan LinearRegression
from sklearn.feature_selection import SelectFromModel

selector = SelectFromModel(estimator=LinearRegression(), prefit=False, threshold="mean")
selector.fit(X_train_transformed, y_train)

selected_features_mask = selector.get_support()
selected_features = X_train_transformed.columns[selected_features_mask].tolist()
print("Fitur terpilih:", selected_features)

feature_importance = pd.DataFrame({
    "Feature": X_train_transformed.columns,
    "Importance": selector.estimator_.coef_,
    "Selected": selected_features_mask,
})
feature_importance.sort_values("Importance", key=abs, ascending=False)

Fitur terpilih: ['Position_Junior', 'Position_Manager']


,Feature,Importance,Selected
8,Position_Manager,-5.000000e-01,True
7,Position_Junior,-4.330127e-01,True
2,Experience_scaled,-1.103284e-15,False
1,Salary_scaled,1.054712e-15,False
4,Department_HR,-4.649059e-16,False
3,Department_Finance,4.614364e-16,False
6,Department_Sales,-2.810252e-16,False
0,Age_scaled,2.078464e-16,False
5,Department_IT,-2.046974e-16,False


**Interpretasi:** tabel menampilkan koefisien tiap fitur (`Importance`) dan apakah fitur itu terpilih (`Selected`). Fitur dengan |koefisien| di atas rata-rata dipertahankan. Parameter seperti `threshold` dapat diubah sesuai kebutuhan.

### Ringkasan: kapan memakai transformasi apa
| Situasi data | Teknik yang cocok |
|---|---|
| Nilai hilang sedikit, numerik | `SimpleImputer` (mean/median) |
| Nilai hilang sedang, hubungan antarfitur penting | `KNNImputer` / `IterativeImputer` |
| Skala fitur berbeda, model berbasis jarak/gradien | `StandardScaler` / `MinMaxScaler` |
| Ada outlier | `RobustScaler` (disebut di bagian pembuka buku) |
| Normalisasi per sampel / data sparse | `Normalizer` |
| Kategori nominal | `OneHotEncoder` |
| Kategori ordinal | `OrdinalEncoder` |
| Data campuran numerik + kategorikal | `ColumnTransformer` |
| Model pohon (Decision Tree / Random Forest) | Scaling umumnya **tidak diperlukan** |

## 6. Practical Exercises: Comprehensive Pipeline pada California Housing
**Tugas dari buku:** (1) muat dataset California Housing, (2) split data, (3) buat pipeline dengan minimal 3 langkah termasuk estimator di akhir, (4) fit pipeline, (5) evaluasi pada data uji.

Dataset: 20.640 sampel dengan fitur numerik tentang blok sensus di California. Target `MedHouseVal` adalah **median nilai rumah** dalam satuan 100.000 USD. (Buku menyebut 9 fitur; yang dimuat `fetch_california_housing` berisi 8 fitur ditambah 1 kolom target.)

> **Perlu koneksi internet:** `fetch_california_housing()` mengunduh dataset pada pemakaian pertama (lalu di-*cache*). Solusi di bawah adalah solusi saya sendiri dan boleh berbeda dari solusi di repo buku.

In [35]:
# [Latihan] 1. Muat dataset
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing(as_frame=True)
housing_X, housing_y = housing.data, housing.target
print(housing_X.shape, housing_y.name)
housing_X.head()

(20640, 8) MedHouseVal


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [36]:
# [Latihan] 2. Split data (sebelum transformasi apa pun, untuk menghindari data leakage)
from sklearn.model_selection import train_test_split

Xh_train, Xh_test, yh_train, yh_test = train_test_split(housing_X, housing_y, test_size=0.2, random_state=2024)
print("Train:", Xh_train.shape, "| Test:", Xh_test.shape)

Train: (16512, 8) | Test: (4128, 8)


In [37]:
# [Latihan] 3. Pipeline: imputer -> scaler -> polynomial features -> Ridge regression
from sklearn.linear_model import Ridge

housing_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("model", Ridge(alpha=1.0)),
])

# 4. Fit
housing_pipeline.fit(Xh_train, yh_train)
housing_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. I

In [38]:
# [Latihan] 5. Evaluasi pada data uji (R^2 dan RMSE), dibandingkan dengan baseline LinearRegression polos
from sklearn.metrics import r2_score, mean_squared_error

pred = housing_pipeline.predict(Xh_test)
print("Pipeline  -> R^2 :", round(r2_score(yh_test, pred), 4),
      "| RMSE :", round(float(np.sqrt(mean_squared_error(yh_test, pred))), 4))

baseline = LinearRegression().fit(Xh_train, yh_train)
pred_b = baseline.predict(Xh_test)
print("Baseline  -> R^2 :", round(r2_score(yh_test, pred_b), 4),
      "| RMSE :", round(float(np.sqrt(mean_squared_error(yh_test, pred_b))), 4))

Pipeline  -> R^2 : 0.6321 | RMSE : 0.7094
Baseline  -> R^2 : 0.5987 | RMSE : 0.7409


**Cara membaca hasil:** $R^2$ mendekati 1 berarti model menjelaskan sebagian besar variasi target, dan RMSE adalah rata-rata kesalahan dalam satuan target (di sini ×100.000 USD). Perbandingan dengan baseline menunjukkan apakah preprocessing dan fitur polinomial benar-benar membantu. Seluruh transformasi hanya dipelajari dari data latih, sehingga skor uji tetap jujur.

> Buku menyebut bahwa **cara mengevaluasi** pipeline tidak dibahas eksplisit di chapter ini. `r2_score` dan `mean_squared_error` dari `sklearn.metrics` adalah metrik standar untuk regresi, dan akan dibahas lebih dalam di bab evaluasi model.

## Kesimpulan Chapter 2
- **Kualitas data menentukan kualitas model.** Preprocessing adalah proses berkelanjutan, bukan langkah sekali jalan.
- **Missing data:** `SimpleImputer` (cepat, sederhana), `KNNImputer` (memanfaatkan tetangga), `IterativeImputer` (regresi antarfitur). Pilih sesuai proporsi dan pola data hilang.
- **Scaling:** `StandardScaler` (mean 0, std 1), `MinMaxScaler` (rentang 0–1), `Normalizer` (norma tiap baris = 1). Model pohon umumnya tidak membutuhkannya.
- **Encoding:** `OneHotEncoder` untuk nominal, `OrdinalEncoder` untuk ordinal; `LabelEncoder` sebaiknya hanya untuk target. `ColumnTransformer` menggabungkan preprocessing berbeda per kolom.
- **Pipeline** menjamin transformasi konsisten, mencegah *data leakage* (split dulu, `fit` hanya pada train), dan memudahkan tuning.
- **Feature engineering:** `PolynomialFeatures` dan `KBinsDiscretizer` membuat fitur baru; `RFE` dan `SelectFromModel` memilih fitur terpenting.